# Arreglo del loop v3: ¿una intensidad más fuerte ayuda?

En v3, WSI eligió sus 3 capas (11, 15 y 7) y la intensidad ganadora fue **α 6**, la más alta de las que se probaron (3, 4, 5, 6). Puede que una más fuerte sea mejor.

**Qué hace:** prueba **α 8 y α 10** en las mismas 3 capas, con los mismos 64 problemas de desarrollo a 16k tokens. Elige al ganador **entre todo lo probado** (v2, v3 y estos dos).
- **Si α 6 sigue ganando:** no gasta más y lo dice (~20 min).
- **Si gana α 8 o 10:** ese arreglo va al test (200 problemas, siempre activo y vía MoI), comparado contra v3 y p50c (~1 h).

**Reusa tu Drive** (`wsi_runs/discover_loop_fix`): no repite nada de lo anterior. Ahora cada paso **guarda cuánto tardó**.

**Uso:** `L4` → `Run all` → acepta Drive con todas las casillas. Al final se descarga `results_discover_v4.json` y el runtime se apaga **solo si el resultado está en Drive**. La celda de apagado ya no depende de las de arriba, así que funciona aunque el runtime se haya reiniciado.

In [ ]:
N_ELICIT, ELICIT_MAX = 300, 8192    # igual que las corridas anteriores (mismos problemas)
N_DEV, DEV_MAX       = 64, 8192
N_TEST, MAX_NEW      = 200, 16384
ALPHAS4 = [8, 10]                   # intensidades nuevas para las capas de v3
W = "/content/work"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q vllm
import vllm, transformers; print("vllm", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
# Progreso en Google Drive: lo que ya terminó no se repite aunque se reinicie el runtime
import os, glob, shutil, json
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/wsi_runs/discover_loop_fix"
os.makedirs(DRIVE, exist_ok=True)
os.makedirs(W, exist_ok=True)
GUARDAR = ("*.json", "*.jsonl", "*.npz", "*.txt")   # solo resultados chicos; los modelos se rehacen en minutos

def guardar():
    for pat in GUARDAR:
        for f in glob.glob(f"{W}/{pat}"):
            shutil.copy2(f, DRIVE)

def restaurar():
    for pat in GUARDAR:
        for f in glob.glob(f"{DRIVE}/{pat}"):
            shutil.copy2(f, W)

def hecho(*archivos):
    return all(os.path.exists(f"{DRIVE}/{a}") for a in archivos)

print("ya guardado en Drive:", sorted(os.listdir(DRIVE)) or "nada todavía (empieza de cero)")

In [ ]:
%%writefile /content/moi_loop_router_vllm.py
"""MoI loop router on Linux GPUs with vLLM (Colab / Kaggle / RunPod): base vs fixed p50c vs MoI, 32,768-token budget.

Same question as `experiment_moi_loop_router.py` (HF, Windows), at the full think35 budget of the p50c evaluation:
thinking on, T=0.6, top_p=0.95, top_k=20, presence_penalty=1.5, max 32,768 new tokens.

Arms (same prompts, same per-request sampling seed):
  * base  : unedited Qwen3.5-0.8B.
  * fixed : the official p50c checkpoint (edit always on).
  * moi   : base until the router fires, then the p50c checkpoint continues the SAME sequence.
            The router is causal, so the MoI trajectory equals the base trajectory up to the firing token; MoI is
            therefore built from the base outputs: cut each base response at the firing point and let the edited
            model continue it. Sequences where the router never fires are identical to base.

Router (Barry's loop definition): dup[t] = 1 if the 16-token window ending at t occurred earlier in the response.
Fires at the first response length n with >= 50% dup over the last 64 tokens.
Loop failure (all arms, applied after generation): the first point where >= 90% of the last 768 tokens are dup.
The sequence is cut there and counted wrong. This is the same stopping rule the HF script used.

Two differences from the HF hook version, both small and stated in the report:
  1. On the switch, vLLM re-encodes the prefix with the edited weights. The HF hook kept the base-weight KV cache for
     the prefix.
  2. vLLM's presence penalty only sees tokens generated in the current request. For the MoI continuation, every
     distinct token of the prefix gets logit_bias -1.5 to keep the penalty. A prefix token that reappears in the
     continuation is therefore penalised twice (-3.0).

Pipeline (each GPU step runs in its own process, so vLLM frees the GPU cleanly):
    python moi_loop_router_vllm.py prepare --work W --edit-dir REPO/benchmarks/exp8_math [--base-dir D] [--n 200]
    python moi_loop_router_vllm.py gen --work W --arm base
    python moi_loop_router_vllm.py gen --work W --arm fixed
    python moi_loop_router_vllm.py plan-moi --work W
    python moi_loop_router_vllm.py gen --work W --arm moi
    python moi_loop_router_vllm.py report --work W [--ppl]
"""
from __future__ import annotations

import argparse
import hashlib
import json
import math
import random
import re
import shutil
import time
from pathlib import Path

import numpy as np

MAX_NEW = 32_768
K_DUP = 16
ROUTER_WINDOW, ROUTER_FRAC = 64, 0.5
KILL_WINDOW, KILL_FRAC = 768, 0.9
SAMPLING = dict(temperature=0.6, top_p=0.95, top_k=20, presence_penalty=1.5)
INSTRUCTION = "Please reason step by step, and put your final answer within \\boxed{}."
BASE_REPO = "Qwen/Qwen3.5-0.8B"


# ------------------------------------------------------------------ io helpers
def read_jsonl(p: Path) -> list[dict]:
    return [json.loads(line) for line in p.read_text(encoding="utf-8").splitlines() if line.strip()]


def write_jsonl(p: Path, rows: list[dict]) -> None:
    p.write_text("".join(json.dumps(r) + "\n" for r in rows), encoding="utf-8")


# ------------------------------------------------------------------ loop statistics
def dup_flags(ids: list[int], k: int = K_DUP) -> list[int]:
    seen, out = set(), [0] * len(ids)
    for t in range(k - 1, len(ids)):
        w = tuple(ids[t - k + 1 : t + 1])
        out[t] = int(w in seen)
        seen.add(w)
    return out


def first_window_hit(d: list[int], window: int, frac: float) -> int | None:
    """Smallest n (response length) such that the last `window` of d[:n] has >= frac dup; None if never."""
    need = frac * window
    s = sum(d[:window])
    if len(d) < window:
        return None
    if s >= need:
        return window
    for n in range(window + 1, len(d) + 1):
        s += d[n - 1] - d[n - 1 - window]
        if s >= need:
            return n
    return None


class LoopKillState:
    """Per-request early stop: forces EOS as soon as the kill rule (>= KILL_FRAC repeated K_DUP-grams over the last
    KILL_WINDOW response tokens) holds. Grading cuts a response at exactly that point anyway (grade -> analyse ->
    first_window_hit), so stopping there gives the same graded result without generating the doomed loop to the cap.
    Incremental: O(1) per new token."""

    def __init__(self, eos_id: int, window: int = KILL_WINDOW, frac: float = KILL_FRAC, k: int = K_DUP):
        self.eos, self.window, self.need, self.k = eos_id, window, frac * window, k
        self.seen: set = set()
        self.flags: list[int] = []
        self.s = 0

    def killed(self, output_ids) -> bool:
        while len(self.flags) < len(output_ids):
            t = len(self.flags)
            f = 0
            if t >= self.k - 1:
                w = tuple(output_ids[t - self.k + 1 : t + 1])
                f = int(w in self.seen)
                self.seen.add(w)
            self.flags.append(f)
            self.s += f
            if t >= self.window:
                self.s -= self.flags[t - self.window]
        return len(self.flags) >= self.window and self.s >= self.need

    def __call__(self, output_ids, logits):
        if self.killed(output_ids):
            logits = logits.clone()
            logits.fill_(float("-inf"))
            logits[self.eos] = 0.0
        return logits


try:  # vLLM V1 per-request logits processors (absent locally / in old vLLM: early stop is then skipped)
    from vllm.v1.sample.logits_processor import AdapterLogitsProcessor as _AdapterLP
except Exception:  # noqa: BLE001
    _AdapterLP = None

if _AdapterLP is not None:
    class LoopKillLogitsProcessor(_AdapterLP):
        """Enabled per request with SamplingParams(extra_args={"loop_kill_eos": <eos id>})."""

        def is_argmax_invariant(self) -> bool:
            return False

        def new_req_logits_processor(self, params):
            ea = getattr(params, "extra_args", None) or {}
            return LoopKillState(int(ea["loop_kill_eos"])) if "loop_kill_eos" in ea else None
else:
    LoopKillLogitsProcessor = None


def strip_eos(ids: list[int], eos: set[int]) -> list[int]:
    for i, t in enumerate(ids):
        if t in eos:
            return ids[:i]
    return ids


# ------------------------------------------------------------------ grading
def extract_answer(text: str) -> str | None:
    if "</think>" in text:
        text = text.split("</think>")[-1]
    boxed = re.findall(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", text)
    cand = boxed[-1] if boxed else text
    nums = re.findall(r"-?\d+(?:\.\d+)?", cand.replace(",", ""))
    return nums[-1] if nums else None


def correct(pred: str | None, gold: str) -> bool:
    try:
        return pred is not None and abs(float(pred) - float(gold)) < 1e-6
    except ValueError:
        return False


# ------------------------------------------------------------------ prepare
def int16_sha(t) -> str:
    import torch

    return hashlib.sha256(t.contiguous().view(torch.int16).numpy().tobytes()).hexdigest()


def build_p50c(base_dir: Path, edit_dir: Path, out_dir: Path) -> dict:
    """Copy the base checkpoint and write the official p50c tensors into it, verifying both SHA-256s."""
    import torch
    from safetensors import safe_open
    from safetensors.torch import save_file

    man = json.loads((edit_dir / "p50c_manifest.json").read_text())
    z = np.load(edit_dir / "p50c_edit.npz", allow_pickle=False)
    if out_dir.exists():
        shutil.rmtree(out_dir)
    shutil.copytree(base_dir, out_dir, ignore=shutil.ignore_patterns(".cache", "*.lock"))
    idx_file = next(iter(sorted(base_dir.glob("*.safetensors.index.json"))), None)
    weight_map = json.loads(idx_file.read_text())["weight_map"] if idx_file else None
    by_file: dict[str, list[int]] = {}
    for i, t in enumerate(man["tensors"]):
        fn = weight_map[t["name"]] if weight_map else next(iter(sorted(base_dir.glob("*.safetensors")))).name
        by_file.setdefault(fn, []).append(i)
    report = {}
    for fn, ids in by_file.items():
        with safe_open(str(base_dir / fn), "pt") as f:
            tensors = {k: f.get_tensor(k) for k in f.keys()}
            meta = f.metadata() or {"format": "pt"}
        for i in ids:
            t = man["tensors"][i]
            base = tensors[t["name"]]
            sha = int16_sha(base)
            if sha != t["base_sha256"]:
                raise SystemExit(f"{t['name']}: base weights differ from the p50c release ({sha[:12]}...)")
            bits = base.view(torch.int16).flatten().clone()
            vals = z[f"t{i}_val"].astype(np.uint16).view(np.int16)
            bits[torch.from_numpy(z[f"t{i}_idx"].astype(np.int64))] = torch.from_numpy(vals)
            edited = bits.view(torch.bfloat16).reshape(base.shape)
            esha = int16_sha(edited)
            ok = esha == t.get("edited_sha256", esha)
            report[t["name"]] = {"base_sha_ok": True, "edited_sha_ok": ok,
                                 "changed": int((edited.view(torch.int16) != base.view(torch.int16)).sum())}
            if not ok:
                raise SystemExit(f"{t['name']}: rebuilt edit does not match the release SHA ({esha[:12]}...)")
            tensors[t["name"]] = edited
        save_file(tensors, str(out_dir / fn), metadata=meta)
    return report


def fetch_datasets(work: Path) -> tuple[Path, Path]:
    from huggingface_hub import hf_hub_download

    gsm = Path(hf_hub_download("openai/gsm8k", "main/test-00000-of-00001.parquet", repo_type="dataset",
                               local_dir=work / "data"))
    wt = Path(hf_hub_download("Salesforce/wikitext", "wikitext-2-raw-v1/test-00000-of-00001.parquet",
                              repo_type="dataset", local_dir=work / "data"))
    return gsm, wt


def eos_ids(tok) -> list[int]:
    ids = {tok.eos_token_id, tok.pad_token_id}
    for s in ("<|im_end|>", "<|endoftext|>"):
        i = tok.convert_tokens_to_ids(s)
        if isinstance(i, int) and i != tok.unk_token_id:
            ids.add(i)
    ids.discard(None)
    return sorted(ids)


def cmd_prepare(a) -> None:
    import pandas as pd
    from transformers import AutoTokenizer

    work = Path(a.work)
    work.mkdir(parents=True, exist_ok=True)
    if a.base_dir:
        base_dir = Path(a.base_dir)
    else:
        from huggingface_hub import snapshot_download

        base_dir = Path(snapshot_download(BASE_REPO, local_dir=work / "qwen-base"))
    p50c_dir = work / "qwen-p50c"
    t0 = time.time()
    rep = build_p50c(base_dir, Path(a.edit_dir), p50c_dir)
    print(f"p50c checkpoint built in {time.time() - t0:.0f}s:", json.dumps(rep))
    if a.gsm and a.wikitext:
        gsm_p, wt_p = Path(a.gsm), Path(a.wikitext)
    else:
        gsm_p, wt_p = fetch_datasets(work)
    tok = AutoTokenizer.from_pretrained(base_dir)
    gsm = pd.read_parquet(gsm_p).iloc[: a.n]
    reqs = []
    for i, (q, ans) in enumerate(zip(gsm["question"], gsm["answer"])):
        ids = tok.apply_chat_template([{"role": "user", "content": q + "\n" + INSTRUCTION}], tokenize=True,
                                      add_generation_prompt=True, enable_thinking=True)
        if hasattr(ids, "keys"):  # BatchEncoding in newer transformers
            ids = ids["input_ids"]
        reqs.append({"id": i, "prompt_ids": list(map(int, ids)), "gold": ans.split("####")[-1].strip().replace(",", ""),
                     "seed": a.seed * 100_003 + i, "max_tokens": a.max_new})
    write_jsonl(work / "requests_base.jsonl", reqs)
    write_jsonl(work / "requests_fixed.jsonl", reqs)
    cfg = {"base_dir": str(base_dir), "p50c_dir": str(p50c_dir), "wikitext": str(wt_p), "n": a.n,
           "max_new": a.max_new, "eos": eos_ids(tok), "sampling": SAMPLING, "seed": a.seed,
           "router": [ROUTER_WINDOW, ROUTER_FRAC], "kill": [KILL_WINDOW, KILL_FRAC], "k_dup": K_DUP}
    (work / "config.json").write_text(json.dumps(cfg, indent=2))
    print(f"{len(reqs)} GSM8K requests, longest prompt {max(len(r['prompt_ids']) for r in reqs)} tokens -> {work}")


# ------------------------------------------------------------------ generation (vLLM)
def cmd_gen(a) -> None:
    from vllm import LLM, SamplingParams
    from vllm.inputs import TokensPrompt

    work = Path(a.work)
    cfg = json.loads((work / "config.json").read_text())
    reqs = read_jsonl(work / f"requests_{a.arm}.jsonl")
    out_p = work / f"outputs_{a.arm}.jsonl"
    if not reqs:
        write_jsonl(out_p, [])
        print("no requests")
        return
    model = cfg["base_dir"] if a.arm == "base" else cfg["p50c_dir"]
    max_len = max(len(r["prompt_ids"]) + r["max_tokens"] for r in reqs) + 16
    llm = LLM(model=model, dtype="bfloat16", max_model_len=max_len, gpu_memory_utilization=a.gpu_mem,
              enable_prefix_caching=True, seed=cfg["seed"])
    sps = [SamplingParams(**SAMPLING, max_tokens=r["max_tokens"], seed=r["seed"], stop_token_ids=cfg["eos"],
                          logit_bias={int(k): float(v) for k, v in r.get("logit_bias", {}).items()} or None)
           for r in reqs]
    t0 = time.time()
    outs = llm.generate([TokensPrompt(prompt_token_ids=r["prompt_ids"]) for r in reqs], sps, use_tqdm=True)
    dt = time.time() - t0
    rows = [{"id": r["id"], "token_ids": list(o.outputs[0].token_ids), "finish_reason": o.outputs[0].finish_reason}
            for r, o in zip(reqs, outs)]
    write_jsonl(out_p, rows)
    ntok = sum(len(x["token_ids"]) for x in rows)
    print(f"[{a.arm}] {len(rows)} sequences, {ntok:,} tokens in {dt / 60:.1f} min ({ntok / dt:,.0f} tok/s) -> {out_p}")


# ------------------------------------------------------------------ MoI plan
def analyse(resp: list[int], eos: set[int]) -> dict:
    """Loop statistics of one response: effective length after the kill rule, and the router firing point."""
    body = strip_eos(resp, eos)
    d = dup_flags(body)
    kill = first_window_hit(d, KILL_WINDOW, KILL_FRAC)
    end = kill if kill is not None else len(body)
    fire = first_window_hit(d[:end], ROUTER_WINDOW, ROUTER_FRAC)
    return {"body": body, "kill": kill, "fire": fire}


def cmd_plan_moi(a) -> None:
    work = Path(a.work)
    cfg = json.loads((work / "config.json").read_text())
    eos = set(cfg["eos"])
    reqs = {r["id"]: r for r in read_jsonl(work / "requests_base.jsonl")}
    cont = []
    for o in read_jsonl(work / "outputs_base.jsonl"):
        st = analyse(o["token_ids"], eos)
        if st["fire"] is None:
            continue
        prefix = st["body"][: st["fire"]]
        r = reqs[o["id"]]
        cont.append({"id": o["id"], "prompt_ids": r["prompt_ids"] + prefix, "prefix_len": len(prefix),
                     "max_tokens": cfg["max_new"] - len(prefix), "seed": r["seed"] + 7,
                     "logit_bias": {str(t): -SAMPLING["presence_penalty"] for t in sorted(set(prefix))}})
    write_jsonl(work / "requests_moi.jsonl", cont)
    print(f"router fired on {len(cont)}/{len(reqs)} base sequences; continuing those with the p50c weights")


# ------------------------------------------------------------------ report
def grade(resp: list[int], finish: str, gold: str, tok, eos: set[int], max_new: int) -> dict:
    st = analyse(resp, eos)
    killed = st["kill"] is not None
    body = st["body"][: st["kill"]] if killed else st["body"]
    truncated = (not killed) and finish == "length" and len(st["body"]) >= max_new
    pred = extract_answer(tok.decode(body, skip_special_tokens=True))
    return {"pred": pred, "tokens": len(body), "looped_killed": killed, "truncated": truncated,
            "correct": bool(correct(pred, gold) and not killed and not truncated), "fire": st["fire"]}


def paired_ci(a: list[bool], b: list[bool], n_boot: int = 10_000, seed: int = 0) -> dict:
    x = np.array(a, float) - np.array(b, float)
    rng = np.random.default_rng(seed)
    boots = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(1)
    return {"diff": round(float(x.mean()), 4), "ci95": [round(float(np.percentile(boots, 2.5)), 4),
                                                         round(float(np.percentile(boots, 97.5)), 4)],
            "a_only": int(((np.array(a)) & ~np.array(b)).sum()), "b_only": int((~np.array(a) & np.array(b)).sum())}


def wikitext_ppl(cfg: dict, n_tokens: int = 20_000) -> dict:
    """Teacher-forced WikiText-2 perplexity; MoI turns the edit on per position with the same causal router."""
    import pandas as pd
    import torch
    import torch.nn.functional as F
    from safetensors import safe_open
    from transformers import AutoModelForCausalLM, AutoTokenizer

    torch.set_grad_enabled(False)
    dev = torch.device("cuda")
    base_dir, p50c_dir = Path(cfg["base_dir"]), Path(cfg["p50c_dir"])
    tok = AutoTokenizer.from_pretrained(base_dir)
    model = AutoModelForCausalLM.from_pretrained(base_dir, dtype=torch.bfloat16).to(dev).eval()
    layers = next(m for n, m in model.named_modules()
                  if n.endswith("layers") and isinstance(m, torch.nn.ModuleList) and len(m) >= 20)
    state = {"gate": None}
    for f in sorted(p50c_dir.glob("*.safetensors")):
        with safe_open(str(f), "pt") as fe, safe_open(str(base_dir / f.name), "pt") as fb:
            for k in fe.keys():
                m = re.search(r"layers\.(\d+)\.mlp\.down_proj\.weight$", k)
                if not m:
                    continue
                dW = fe.get_tensor(k).float() - fb.get_tensor(k).float()
                cols = torch.nonzero(dW.abs().sum(0) > 0).flatten()
                if not len(cols):
                    continue
                cols_d, dW_d = cols.to(dev), dW[:, cols].to(dev)

                def hook(mod, inp, out, cols_d=cols_d, dW_d=dW_d):
                    g = state["gate"]
                    if g is None:
                        return out
                    return (out.float() + (inp[0][..., cols_d].float() @ dW_d.T) * g).to(out.dtype)

                layers[int(m.group(1))].mlp.down_proj.register_forward_hook(hook)
    wt = "\n".join(pd.read_parquet(cfg["wikitext"])["text"].tolist())
    ids_all = tok(wt, add_special_tokens=False).input_ids[:n_tokens]
    res = {}
    for arm in ("base", "fixed", "moi"):
        nll, nt, gated = 0.0, 0, 0
        for i in range(0, len(ids_all) - 16, 512):
            ch = ids_all[i : i + 512]
            ids = torch.tensor([ch], device=dev)
            if arm == "base":
                state["gate"] = None
            elif arm == "fixed":
                state["gate"] = torch.ones(1, 1, 1, device=dev)
            else:
                g = torch.zeros(1, len(ch), 1, device=dev)
                n = first_window_hit(dup_flags(ch), ROUTER_WINDOW, ROUTER_FRAC)
                if n is not None:
                    g[0, n - 1 :] = 1.0
                gated += int(g.sum())
                state["gate"] = g
            lp = F.log_softmax(model(input_ids=ids).logits[0, :-1].float(), dim=-1)
            nll -= float(lp.gather(1, ids[0, 1:, None]).sum())
            nt += len(ch) - 1
        res[arm] = {"ppl": round(math.exp(nll / nt), 4), "tokens": nt, "gated_positions": gated}
        print(f"  WikiText ppl [{arm}] = {res[arm]['ppl']}")
    state["gate"] = None
    return res


def cmd_report(a) -> None:
    from transformers import AutoTokenizer

    work = Path(a.work)
    cfg = json.loads((work / "config.json").read_text())
    eos, max_new = set(cfg["eos"]), cfg["max_new"]
    tok = AutoTokenizer.from_pretrained(cfg["base_dir"])
    reqs = {r["id"]: r for r in read_jsonl(work / "requests_base.jsonl")}
    outs = {arm: {o["id"]: o for o in read_jsonl(work / f"outputs_{arm}.jsonl")} for arm in ("base", "fixed")}
    cont = {o["id"]: o for o in read_jsonl(work / "outputs_moi.jsonl")} if (work / "outputs_moi.jsonl").exists() else {}
    plan = {r["id"]: r for r in read_jsonl(work / "requests_moi.jsonl")} if cont else {}

    rows: dict[str, list[dict]] = {"base": [], "fixed": [], "moi": []}
    for i in sorted(reqs):
        gold = reqs[i]["gold"]
        for arm in ("base", "fixed"):
            o = outs[arm][i]
            rows[arm].append({"id": i, "gold": gold, **grade(o["token_ids"], o["finish_reason"], gold, tok, eos, max_new),
                              "edit_share": 0.0 if arm == "base" else 1.0})
        if i in cont:
            p = plan[i]
            prefix = p["prompt_ids"][len(reqs[i]["prompt_ids"]) :]
            g = grade(prefix + cont[i]["token_ids"], cont[i]["finish_reason"], gold, tok, eos, max_new)
            g["fire"] = p["prefix_len"]
            g["edit_share"] = round((g["tokens"] - p["prefix_len"]) / max(g["tokens"], 1), 4)
        else:  # router never fired: MoI is the base sequence
            g = {**rows["base"][-1], "fire": None, "edit_share": 0.0}
            g = {k: v for k, v in g.items() if k not in ("id", "gold")}
        rows["moi"].append({"id": i, "gold": gold, **g})

    def summ(rs: list[dict]) -> dict:
        n = len(rs)
        return {"n": n, "accuracy": round(sum(r["correct"] for r in rs) / n, 4),
                "loop_killed": round(sum(r["looped_killed"] for r in rs) / n, 4),
                "truncated": round(sum(r["truncated"] for r in rs) / n, 4),
                "mean_tokens": round(sum(r["tokens"] for r in rs) / n, 1),
                "edit_token_share": round(sum(r["edit_share"] * r["tokens"] for r in rs)
                                          / max(sum(r["tokens"] for r in rs), 1), 4)}

    result = {"config": cfg, "summary": {arm: summ(rs) for arm, rs in rows.items()}}
    if len(reqs) > 48:
        result["summary_first48"] = {arm: summ(rs[:48]) for arm, rs in rows.items()}
    acc = {arm: [r["correct"] for r in rs] for arm, rs in rows.items()}
    result["paired"] = {"moi_vs_base": paired_ci(acc["moi"], acc["base"]),
                        "fixed_vs_base": paired_ci(acc["fixed"], acc["base"]),
                        "moi_vs_fixed": paired_ci(acc["moi"], acc["fixed"])}
    fired = [r for r in rows["moi"] if r["fire"] is not None]
    result["router"] = {"fired_share": round(len(fired) / len(rows["moi"]), 4),
                        "median_fire_token": float(np.median([r["fire"] for r in fired])) if fired else None,
                        "never_fired_correct": sum(r["correct"] for r in rows["moi"] if r["fire"] is None),
                        "never_fired": sum(r["fire"] is None for r in rows["moi"])}
    result["caveats"] = ["MoI prefix is re-encoded with the edited weights at the switch (HF hook kept base KV).",
                         "MoI continuation: prefix tokens get logit_bias -1.5; a prefix token reused later is "
                         "penalised -3.0 instead of -1.5."]
    if a.ppl:
        result["wikitext"] = wikitext_ppl(cfg)
    result["rows"] = rows

    print(f"\n{'arm':6s} {'acc':>7s} {'loops':>7s} {'trunc':>7s} {'tokens':>8s} {'edit':>6s}"
          + (f" {'ppl':>8s}" if a.ppl else ""))
    for arm, s in result["summary"].items():
        print(f"{arm:6s} {s['accuracy']:7.1%} {s['loop_killed']:7.1%} {s['truncated']:7.1%} {s['mean_tokens']:8.0f} "
              f"{s['edit_token_share']:6.1%}" + (f" {result['wikitext'][arm]['ppl']:8.3f}" if a.ppl else ""))
    for k, v in result["paired"].items():
        print(f"  {k}: {v['diff']:+.1%}  95% CI [{v['ci95'][0]:+.1%}, {v['ci95'][1]:+.1%}]"
              f"  (only first {v['a_only']}, only second {v['b_only']})")
    print("  router:", result["router"])
    dst = work / "results_32k.json"
    dst.write_text(json.dumps(result, indent=2))
    print(f"\nSaved {dst}")


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("prepare")
    p.add_argument("--work", required=True)
    p.add_argument("--edit-dir", required=True)
    p.add_argument("--base-dir", default=None, help="local Qwen3.5-0.8B; downloaded from the Hub if omitted")
    p.add_argument("--gsm", default=None)
    p.add_argument("--wikitext", default=None)
    p.add_argument("--n", type=int, default=200)
    p.add_argument("--max-new", type=int, default=MAX_NEW)
    p.add_argument("--seed", type=int, default=0)
    g = sub.add_parser("gen")
    g.add_argument("--work", required=True)
    g.add_argument("--arm", choices=["base", "fixed", "moi"], required=True)
    g.add_argument("--gpu-mem", type=float, default=0.88)
    m = sub.add_parser("plan-moi")
    m.add_argument("--work", required=True)
    r = sub.add_parser("report")
    r.add_argument("--work", required=True)
    r.add_argument("--ppl", action="store_true")
    a = ap.parse_args()
    random.seed(0)
    {"prepare": cmd_prepare, "gen": cmd_gen, "plan-moi": cmd_plan_moi, "report": cmd_report}[a.cmd](a)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/discover_loop_fix.py
"""Can WSI find a reasoning-loop fix in Qwen3.5-0.8B on its own, from the BASE model only?

Nothing from the MIT CSAIL p50c release is used to build the fix. p50c is loaded only at the very end, as an exam:
does the discovered fix work as well, and did it find the same layers / direction / neurons?

Stages (each GPU stage runs in its own process; vLLM for generation, transformers for capture):
  elicit    let the base model reason on GSM8K TRAIN problems (think35 sampling) and record which outputs loop.
  diagnose  capture the MLP outputs and hidden units on looping vs normal response tokens (split in two halves):
              * direction u_l = mean MLP output on loop tokens - on normal tokens (half A), per layer
              * separation d'_l of the projection on u_l, measured on half B (held out)
              * choose the layers with the largest d'; in them, the neurons whose activation change pushes
                along u most:  c_i = (E[h_i|loop] - E[h_i|normal]) * (u . w_i)
  build     write the fix with the same form as p50c:  W' = W - alpha * u (m * u^T W)   (m = chosen columns)
  gen       generate with vLLM for a list of requests and a checkpoint
  tune      pick alpha on held-out TRAIN problems (accuracy, then fewer loops)
  plan-moi  MoI arm: cut each base test response where the loop router fires; the fixed model continues it
  report    GSM8K TEST: base vs WSI fix always on vs WSI fix via MoI vs p50c, plus the exam against p50c.

Uses helpers from moi_loop_router_vllm.py (same folder). Usage: see notebooks/discover_loop_fix_colab.ipynb.
"""
from __future__ import annotations

import argparse
import json
import re
import shutil
import sys
import time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import moi_loop_router_vllm as lr  # noqa: E402

LOOP_FRAC, LOOP_WINDOW = lr.ROUTER_FRAC, lr.ROUTER_WINDOW  # a token is "looping" when the router would fire there
CLEAN_WINDOW = 256  # a token is "normal" when the last 256 response tokens have no repeated 16-gram


# ------------------------------------------------------------------ timing (saved to work/timing.json, backed up with the results)
def _gpu_name() -> str:
    try:
        import torch

        return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
    except Exception:  # noqa: BLE001
        return "unknown"


def record_time(work: Path, key: str, seconds: float) -> None:
    p = work / "timing.json"
    t = json.loads(p.read_text()) if p.exists() else {}
    t[key] = {"s": round(seconds, 1), "gpu": _gpu_name()}
    p.write_text(json.dumps(t, indent=1))
    print(f"[time] {key}: {seconds / 60:.1f} min", flush=True)


def run_timed(a, fns: dict, key: str | None) -> None:
    """Run one subcommand; unless key is None, store its wall time (model loading included) in timing.json."""
    t0 = time.perf_counter()
    fns[a.cmd](a)
    if key:
        record_time(Path(a.work), key, time.perf_counter() - t0)


DISCOVERY_PREFIXES = ("gen:outputs_elicit", "diagnose", "build:", "gen:outputs_dev", "pick", "sft-data", "train",
                      "export")


def timing_summary(work: Path) -> dict:
    """Raw per-step times plus the cost of FINDING the fix: model reasoning on train problems, diagnosis, building
    each candidate fix and tuning it on dev problems (test-set evaluation and the p50c exam are not counted)."""
    p = work / "timing.json"
    if not p.exists():
        return {}
    t = json.loads(p.read_text())
    disc = {k: v["s"] for k, v in t.items() if k.startswith(DISCOVERY_PREFIXES)}
    return {"steps": t, "discovery_steps": sorted(disc), "discovery_s": round(sum(disc.values()), 1),
            "discovery_min": round(sum(disc.values()) / 60, 1)}


# ------------------------------------------------------------------ helpers
def cfg_load(work: Path) -> dict:
    return json.loads((work / "config.json").read_text())


def cfg_save(work: Path, cfg: dict) -> None:
    (work / "config.json").write_text(json.dumps(cfg, indent=2))


def make_requests(tok, df, seed: int, max_new: int, offset: int = 0) -> list[dict]:
    reqs = []
    for i, (q, a) in enumerate(zip(df["question"], df["answer"])):
        ids = tok.apply_chat_template([{"role": "user", "content": q + "\n" + lr.INSTRUCTION}], tokenize=True,
                                      add_generation_prompt=True, enable_thinking=True)
        if hasattr(ids, "keys"):
            ids = ids["input_ids"]
        reqs.append({"id": offset + i, "prompt_ids": list(map(int, ids)),
                     "gold": a.split("####")[-1].strip().replace(",", ""),
                     "seed": seed * 100_003 + offset + i, "max_tokens": max_new})
    return reqs


def token_labels(body: list[int]) -> np.ndarray:
    """Per response token: 1 = looping (>= 50% dup over the last 64), 0 = normal (no dup in the last 256), -1 = other."""
    d = np.array(lr.dup_flags(body), dtype=np.float64)
    c = np.concatenate([[0.0], np.cumsum(d)])
    n = len(d)
    lab = np.full(n, -1, dtype=np.int64)
    for t in range(n):
        if t + 1 >= LOOP_WINDOW and (c[t + 1] - c[t + 1 - LOOP_WINDOW]) >= LOOP_FRAC * LOOP_WINDOW:
            lab[t] = 1
        elif t + 1 >= CLEAN_WINDOW and c[t + 1] - c[t + 1 - CLEAN_WINDOW] == 0:
            lab[t] = 0
    return lab


def down_key(keys, layer: int) -> str:
    pat = re.compile(rf"(^|\.)layers\.{layer}\.mlp\.down_proj\.weight$")
    hits = [k for k in keys if pat.search(k) and not k.startswith("mtp.")]  # Qwen3.5 also has mtp.layers.0
    if len(hits) != 1:
        raise SystemExit(f"down_proj of layer {layer} not found uniquely: {hits}")
    return hits[0]


# ------------------------------------------------------------------ prepare
def cmd_prepare(a) -> None:
    import pandas as pd
    from transformers import AutoTokenizer

    work = Path(a.work)
    work.mkdir(parents=True, exist_ok=True)
    if a.base_dir:
        base_dir = Path(a.base_dir)
    else:
        from huggingface_hub import snapshot_download

        base_dir = Path(snapshot_download(lr.BASE_REPO, local_dir=work / "qwen-base"))
    from huggingface_hub import hf_hub_download

    def parquet(split: str) -> Path:
        if a.data_dir and (Path(a.data_dir) / f"{split}.parquet").exists():
            return Path(a.data_dir) / f"{split}.parquet"
        return Path(hf_hub_download("openai/gsm8k", f"main/{split}-00000-of-00001.parquet", repo_type="dataset",
                                    local_dir=work / "data"))

    tok = AutoTokenizer.from_pretrained(base_dir)
    train = pd.read_parquet(parquet("train")).sample(frac=1.0, random_state=a.seed).reset_index(drop=True)
    test = pd.read_parquet(parquet("test")).iloc[: a.n_test]
    lr.write_jsonl(work / "requests_elicit.jsonl",
                   make_requests(tok, train.iloc[: a.n_elicit], a.seed, a.elicit_max_new, offset=0))
    lr.write_jsonl(work / "requests_dev.jsonl",
                   make_requests(tok, train.iloc[a.n_elicit : a.n_elicit + a.n_dev], a.seed, a.dev_max_new,
                                 offset=100_000))
    lr.write_jsonl(work / "requests_test.jsonl", make_requests(tok, test, a.seed, a.max_new, offset=0))
    cfg_save(work, {"base_dir": str(base_dir), "eos": lr.eos_ids(tok), "max_new": a.max_new, "seed": a.seed,
                    "n_elicit": a.n_elicit, "n_dev": a.n_dev, "n_test": len(test)})
    print(f"requests: {a.n_elicit} elicit + {a.n_dev} dev (GSM8K train, disjoint) + {len(test)} test -> {work}")


# ------------------------------------------------------------------ generation (vLLM)
def cmd_gen(a) -> None:
    from vllm import LLM, SamplingParams
    from vllm.inputs import TokensPrompt

    work = Path(a.work)
    cfg = cfg_load(work)
    reqs = lr.read_jsonl(work / a.requests)
    out_p = work / a.out
    if not reqs:
        lr.write_jsonl(out_p, [])
        return
    max_len = max(len(r["prompt_ids"]) + r["max_tokens"] for r in reqs) + 16
    base_kw = dict(model=a.model, tokenizer=cfg["base_dir"], dtype="bfloat16", max_model_len=max_len,
                   gpu_memory_utilization=a.gpu_mem, enable_prefix_caching=True, seed=cfg["seed"])
    # speed-ups, each dropped if this vLLM rejects it:
    #   text only  -> skips the vision encoder's profiling / warm-up (Qwen3.5 is multimodal; we never send images)
    #   loop kill  -> stops a response where the kill rule fires instead of generating the dead loop to the cap
    extras = {"limit_mm_per_prompt": {"image": 0, "video": 0}}
    use_kill = a.early_stop and lr.LoopKillLogitsProcessor is not None
    if use_kill:
        extras["logits_processors"] = [lr.LoopKillLogitsProcessor]
    llm = None
    for drop in ([], ["limit_mm_per_prompt"], ["logits_processors"], ["limit_mm_per_prompt", "logits_processors"]):
        kw = {**base_kw, **{k: v for k, v in extras.items() if k not in drop}}
        try:
            llm = LLM(**kw)
            use_kill = use_kill and "logits_processors" in kw
            print(f"[gen] vLLM options: text_only={'limit_mm_per_prompt' in kw} early_loop_stop={use_kill}", flush=True)
            break
        except Exception as e:  # noqa: BLE001
            print(f"[gen] vLLM refused {sorted(set(extras) - set(drop))}: {type(e).__name__}: {str(e)[:200]}", flush=True)
            import gc

            gc.collect()
    if llm is None:
        raise SystemExit("vLLM could not start")
    kill_eos = cfg["eos"][-1]

    def params(r, kill: bool):
        return SamplingParams(**lr.SAMPLING, max_tokens=r["max_tokens"], seed=r["seed"], stop_token_ids=cfg["eos"],
                              logit_bias={int(k): float(v) for k, v in r.get("logit_bias", {}).items()} or None,
                              extra_args={"loop_kill_eos": kill_eos} if kill else None)

    prompts = [TokensPrompt(prompt_token_ids=r["prompt_ids"]) for r in reqs]
    t0 = time.time()
    try:
        outs = llm.generate(prompts, [params(r, use_kill) for r in reqs], use_tqdm=True)
    except Exception as e:  # noqa: BLE001
        if not use_kill:
            raise
        print(f"[gen] early loop stop failed ({type(e).__name__}: {str(e)[:200]}); generating without it", flush=True)
        use_kill = False
        outs = llm.generate(prompts, [params(r, False) for r in reqs], use_tqdm=True)
    dt = time.time() - t0
    rows = [{"id": r["id"], "token_ids": list(o.outputs[0].token_ids), "finish_reason": o.outputs[0].finish_reason}
            for r, o in zip(reqs, outs)]
    lr.write_jsonl(out_p, rows)
    ntok = sum(len(x["token_ids"]) for x in rows)
    stopped = 0
    if use_kill:  # sanity: responses ended by the forced EOS = the kill rule holds right before the final token
        for x in rows:
            ids = x["token_ids"]
            if ids and ids[-1] == kill_eos and lr.LoopKillState(kill_eos).killed(ids[:-1]):
                stopped += 1
    print(f"[gen {a.out}] {len(rows)} seqs, {ntok:,} tokens, {dt / 60:.1f} min ({ntok / dt:,.0f} tok/s)"
          + (f"; {stopped} loops stopped early" if use_kill else ""))


def score(work: Path, req_file: str, out_file: str, tok=None) -> dict:
    from transformers import AutoTokenizer

    cfg = cfg_load(work)
    tok = tok or AutoTokenizer.from_pretrained(cfg["base_dir"])
    reqs = {r["id"]: r for r in lr.read_jsonl(work / req_file)}
    rows = []
    for o in lr.read_jsonl(work / out_file):
        r = reqs[o["id"]]
        rows.append({"id": o["id"], **lr.grade(o["token_ids"], o["finish_reason"], r["gold"], tok, set(cfg["eos"]),
                                                r["max_tokens"])})
    n = max(len(rows), 1)
    return {"n": len(rows), "accuracy": sum(x["correct"] for x in rows) / n,
            "loop_killed": sum(x["looped_killed"] for x in rows) / n,
            "truncated": sum(x["truncated"] for x in rows) / n, "rows": rows}


# ------------------------------------------------------------------ diagnose (transformers)
def cmd_diagnose(a) -> None:
    import torch
    from transformers import AutoModelForCausalLM

    work = Path(a.work)
    cfg = cfg_load(work)
    eos = set(cfg["eos"])
    reqs = {r["id"]: r for r in lr.read_jsonl(work / "requests_elicit.jsonl")}
    outs = lr.read_jsonl(work / "outputs_elicit.jsonl")

    # which sequences, and which of their tokens, are looping vs normal
    seqs = []
    n_loop_seq = 0
    for o in outs:
        body = lr.strip_eos(o["token_ids"], eos)
        lab = token_labels(body)
        if not (lab >= 0).any():
            continue
        first_loop = int(np.argmax(lab == 1)) if (lab == 1).any() else None
        end = min(len(body), (first_loop + a.after_onset) if first_loop is not None else a.max_seq, a.max_seq)
        n_loop_seq += first_loop is not None
        seqs.append({"id": o["id"], "prompt": reqs[o["id"]]["prompt_ids"], "body": body[:end], "lab": lab[:end]})
    loop_tok = sum(int((s["lab"] == 1).sum()) for s in seqs)
    norm_tok = sum(int((s["lab"] == 0).sum()) for s in seqs)
    print(f"elicit: {len(outs)} responses, {n_loop_seq} with loops; labelled tokens: {loop_tok:,} looping, "
          f"{norm_tok:,} normal")
    if loop_tok < 500:
        raise SystemExit("too few looping tokens to diagnose; raise --n-elicit or --elicit-max-new")
    rng = np.random.default_rng(cfg["seed"])
    half = rng.permutation(len(seqs)) % 2  # 0 = A (fit), 1 = B (held-out separation)

    dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    dtype = torch.bfloat16 if dev.type == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(cfg["base_dir"], dtype=dtype).to(dev).eval()
    layers = next(m for n, m in model.named_modules()
                  if n.endswith("layers") and isinstance(m, torch.nn.ModuleList) and len(m) >= 8)
    L = len(layers)
    downs = [lyr.mlp.down_proj for lyr in layers]
    U, D = downs[0].in_features, downs[0].out_features
    body_model = getattr(model, "model", model)  # skip the LM head: only internal activations are needed

    cur = {"mask": None}
    acc: dict = {}

    def hook(li):
        def f(mod, inp, out):
            m = cur["mask"]
            h = inp[0][0].float()
            y = out[0].float()
            for cls in (0, 1):
                sel = m == cls
                if not sel.any():
                    continue
                k = (li, cls)
                hs, ys = h[sel], y[sel]
                if cur["pass"] == "A":
                    if k not in acc:
                        acc[k] = {"n": 0, "h": torch.zeros(U, device=dev, dtype=torch.float64),
                                  "y": torch.zeros(D, device=dev, dtype=torch.float64)}
                    acc[k]["n"] += int(sel.sum())
                    acc[k]["h"] += hs.sum(0).double()
                    acc[k]["y"] += ys.sum(0).double()
                else:
                    p = (ys @ cur["u"][li]).double()
                    b = acc.setdefault(("B",) + k, {"n": 0, "s": 0.0, "ss": 0.0})
                    b["n"] += int(sel.sum())
                    b["s"] += float(p.sum())
                    b["ss"] += float((p * p).sum())
        return f

    hs = [d.register_forward_hook(hook(i)) for i, d in enumerate(downs)]

    def run(which: int, name: str) -> None:
        cur["pass"] = name
        t0 = time.perf_counter()
        with torch.no_grad():
            for s, hb in zip(seqs, half):
                if hb != which:
                    continue
                ids = torch.tensor([s["prompt"] + s["body"]], device=dev)
                full = np.full(ids.shape[1], -1, dtype=np.int64)
                full[len(s["prompt"]) :] = s["lab"]
                cur["mask"] = torch.from_numpy(full).to(dev)
                body_model(input_ids=ids, use_cache=False)
        print(f"  pass {name}: {time.perf_counter() - t0:.0f}s", flush=True)

    run(0, "A")
    u = []
    for li in range(L):
        d = acc[(li, 1)]["y"] / acc[(li, 1)]["n"] - acc[(li, 0)]["y"] / acc[(li, 0)]["n"]
        u.append((d / d.norm()).float())
    cur["u"] = u
    run(1, "B")
    for h in hs:
        h.remove()

    sep = []
    for li in range(L):
        st = {}
        for cls in (0, 1):
            b = acc[("B", li, cls)]
            m = b["s"] / b["n"]
            st[cls] = (m, max(b["ss"] / b["n"] - m * m, 1e-12))
        sep.append((st[1][0] - st[0][0]) / np.sqrt((st[1][1] + st[0][1]) / 2))
    order = [int(x) for x in np.argsort(sep)[::-1]]
    # two generic layer candidates (no knowledge of p50c): most separated overall, most separated in the later half;
    # the dev set decides between them in `pick-alpha`
    cands = {"top": order[: a.top_layers], "late": [x for x in order if x >= L // 2][: a.top_layers]}
    # architecture prior (not p50c): in hybrid models, the most separated layers among the full-attention ones
    tc = getattr(model.config, "text_config", None) or model.config
    ltypes = getattr(tc, "layer_types", None)
    if ltypes and len(ltypes) == L:
        fa_all = [x for x in order if "full" in str(ltypes[x])]
        if fa_all:
            cands["fa"] = fa_all[: a.top_layers]
            if a.fa_layers and a.fa_layers != a.top_layers:  # e.g. fa3: the three most separated full-attention layers
                cands[f"fa{a.fa_layers}"] = fa_all[: a.fa_layers]
    chosen = cands["top"]
    print("held-out separation d' per layer:", " ".join(f"L{li}:{s:.2f}" for li, s in enumerate(sep)))
    print("layer candidates:", cands)

    neurons, u_out = {}, {}
    for li in sorted(set().union(*cands.values())):
        dh = acc[(li, 1)]["h"] / acc[(li, 1)]["n"] - acc[(li, 0)]["h"] / acc[(li, 0)]["n"]
        W = downs[li].weight.double()  # [D, U]
        c = dh * (u[li].double() @ W)
        top = c.topk(a.neurons).indices.sort().values
        neurons[str(li)] = top.tolist()
        u_out[str(li)] = u[li].cpu().numpy()
    np.savez(work / "wsi_directions.npz", **{f"u{k}": v for k, v in u_out.items()},
             **{f"all_u{li}": u[li].cpu().numpy() for li in range(L)})
    diag = {"separation": [float(s) for s in sep], "layer_rank": [int(x) for x in order], "chosen_layers": chosen, "candidates": cands,
            "neurons": neurons, "loop_tokens": loop_tok, "normal_tokens": norm_tok, "sequences": len(seqs),
            "looping_sequences": n_loop_seq}
    (work / "diagnosis.json").write_text(json.dumps(diag, indent=2))
    print(f"diagnosis saved: candidates {cands}, {a.neurons} neurons per layer")


# ------------------------------------------------------------------ build the fix
def cmd_build(a) -> None:
    import torch
    from safetensors import safe_open
    from safetensors.torch import save_file

    work = Path(a.work)
    cfg = cfg_load(work)
    base = Path(cfg["base_dir"])
    diag = json.loads((work / "diagnosis.json").read_text())
    z = np.load(work / "wsi_directions.npz")
    if a.cand not in diag["candidates"]:
        raise SystemExit(f"candidate {a.cand} not in diagnosis.json: {sorted(diag['candidates'])}")
    layers_sel = diag["candidates"][a.cand]
    out = Path(a.out_dir) if a.out_dir else work / f"wsi-fix-{a.cand}-a{a.alpha:g}"
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(base, out, ignore=shutil.ignore_patterns(".cache", "*.lock"))
    for f in sorted(base.glob("*.safetensors")):
        with safe_open(str(f), "pt") as fh:
            keys = list(fh.keys())
            targets = {}
            for li in layers_sel:
                try:
                    targets[down_key(keys, li)] = li
                except SystemExit:
                    continue
            if not targets:
                continue
            tensors = {k: fh.get_tensor(k) for k in keys}
            meta = fh.metadata() or {"format": "pt"}
        for k, li in targets.items():
            W = tensors[k].double()
            u = torch.from_numpy(z[f"u{li}"]).double()
            m = torch.zeros(W.shape[1], dtype=torch.float64)
            m[diag["neurons"][str(li)]] = 1.0
            W = W - a.alpha * torch.outer(u, m * (u @ W))
            tensors[k] = W.to(tensors[k].dtype)
        save_file(tensors, str(out / f.name), metadata=meta)
    print(f"built {out} (alpha={a.alpha}, layers {layers_sel})")


# ------------------------------------------------------------------ MoI plan
def cmd_plan_moi(a) -> None:
    work = Path(a.work)
    cfg = cfg_load(work)
    eos = set(cfg["eos"])
    reqs = {r["id"]: r for r in lr.read_jsonl(work / "requests_test.jsonl")}
    cont = []
    for o in lr.read_jsonl(work / "outputs_test_base.jsonl"):
        st = lr.analyse(o["token_ids"], eos)
        if st["fire"] is None:
            continue
        prefix = st["body"][: st["fire"]]
        r = reqs[o["id"]]
        cont.append({"id": o["id"], "prompt_ids": r["prompt_ids"] + prefix, "prefix_len": len(prefix),
                     "max_tokens": r["max_tokens"] - len(prefix), "seed": r["seed"] + 7,
                     "logit_bias": {str(t): -lr.SAMPLING["presence_penalty"] for t in sorted(set(prefix))}})
    lr.write_jsonl(work / "requests_test_moi.jsonl", cont)
    print(f"router fired on {len(cont)}/{len(reqs)} base test responses")


# ------------------------------------------------------------------ tune + report
def cmd_pick_alpha(a) -> None:
    work = Path(a.work)
    cfg = cfg_load(work)
    res = {}
    for f in sorted(work.glob("outputs_dev_*_a*.jsonl")):
        cand, al = f.stem[len("outputs_dev_"):].rsplit("_a", 1)
        s = score(work, "requests_dev.jsonl", f.name)
        res[(cand, float(al))] = {k: v for k, v in s.items() if k != "rows"}
        print(f"  {cand:4s} alpha={float(al):g}: acc={s['accuracy']:.1%} loops={s['loop_killed']:.1%} "
              f"trunc={s['truncated']:.1%}")
    base = score(work, "requests_dev.jsonl", "outputs_dev_base.jsonl")
    print(f"  base     : acc={base['accuracy']:.1%} loops={base['loop_killed']:.1%}")
    best = max(res, key=lambda k: (res[k]["accuracy"], -res[k]["loop_killed"]))
    cfg["cand"], cfg["alpha"] = best
    cfg["dev"] = {"base": {k: v for k, v in base.items() if k != "rows"},
                  "grid": {f"{c}@{al:g}": v for (c, al), v in res.items()}}
    cfg_save(work, cfg)
    diag_p = work / "diagnosis.json"
    diag = json.loads(diag_p.read_text())
    diag["chosen_layers"] = diag["candidates"][best[0]]
    diag_p.write_text(json.dumps(diag, indent=2))
    print(f"chosen: layers {diag['chosen_layers']} ({best[0]}), alpha = {best[1]:g}")
    (work / "chosen_fix.txt").write_text(str(work / f"wsi-fix-{best[0]}-a{best[1]:g}"))


def cmd_report(a) -> None:
    import torch
    from transformers import AutoTokenizer

    work = Path(a.work)
    cfg = cfg_load(work)
    tok = AutoTokenizer.from_pretrained(cfg["base_dir"])
    eos, reqs = set(cfg["eos"]), {r["id"]: r for r in lr.read_jsonl(work / "requests_test.jsonl")}
    arms = {"base": "outputs_test_base.jsonl", "wsi_fixed": "outputs_test_wsi.jsonl", "p50c": "outputs_test_p50c.jsonl"}
    res: dict = {"alpha": cfg.get("alpha"), "layer_candidate": cfg.get("cand"), "dev": cfg.get("dev"), "arms": {}}
    rows = {}
    for name, f in arms.items():
        if (work / f).exists():
            s = score(work, "requests_test.jsonl", f, tok)
            rows[name] = {r["id"]: r for r in s["rows"]}
            res["arms"][name] = {k: v for k, v in s.items() if k != "rows"}
    # MoI = base until the router fires, then the WSI fix continues the same sequence
    plan = {r["id"]: r for r in lr.read_jsonl(work / "requests_test_moi.jsonl")}
    cont = {o["id"]: o for o in lr.read_jsonl(work / "outputs_test_moi.jsonl")} if plan else {}
    moi, edit_tok, all_tok = {}, 0, 0
    for i in reqs:
        if i in cont:
            pre = plan[i]["prompt_ids"][len(reqs[i]["prompt_ids"]) :]
            g = lr.grade(pre + cont[i]["token_ids"], cont[i]["finish_reason"], reqs[i]["gold"], tok, eos,
                         reqs[i]["max_tokens"])
            edit_tok += max(g["tokens"] - len(pre), 0)
        else:
            g = rows["base"][i]
        all_tok += g["tokens"]
        moi[i] = g
    n = len(moi)
    res["arms"]["wsi_moi"] = {"n": n, "accuracy": sum(g["correct"] for g in moi.values()) / n,
                              "loop_killed": sum(g["looped_killed"] for g in moi.values()) / n,
                              "truncated": sum(g["truncated"] for g in moi.values()) / n,
                              "edit_token_share": edit_tok / max(all_tok, 1), "router_fired": len(plan) / n}
    rows["wsi_moi"] = moi
    ids = sorted(reqs)
    acc = {k: [bool(rows[k][i]["correct"]) for i in ids] for k in rows}
    res["paired"] = {f"{x}_vs_{y}": lr.paired_ci(acc[x], acc[y])
                     for x, y in (("wsi_fixed", "base"), ("wsi_moi", "base"), ("wsi_moi", "wsi_fixed"),
                                  ("wsi_fixed", "p50c")) if x in acc and y in acc}

    # ---- the exam: compare with p50c only now
    diag = json.loads((work / "diagnosis.json").read_text())
    exam: dict = {"wsi_chosen_layers": diag["chosen_layers"], "p50c_layers": [15, 19],
                  "rank_of_15_and_19_in_wsi_separation": {str(l): diag["layer_rank"].index(l) + 1 for l in (15, 19)}}
    if a.p50c_dir:
        from safetensors import safe_open

        z = np.load(work / "wsi_directions.npz")
        base = Path(cfg["base_dir"])
        for li in (15, 19):
            for f in sorted(Path(a.p50c_dir).glob("*.safetensors")):
                with safe_open(str(f), "pt") as fe, safe_open(str(base / f.name), "pt") as fb:
                    try:
                        k = down_key(list(fe.keys()), li)
                    except SystemExit:
                        continue
                    dW = (fe.get_tensor(k).double() - fb.get_tensor(k).double()).numpy()
            cols = np.where(np.abs(dW).sum(0) > 0)[0]
            Up, _, _ = np.linalg.svd(dW[:, cols], full_matrices=False)
            up = Up[:, 0]
            uw = z[f"all_u{li}"].astype(np.float64)
            rec = {"abs_cos_wsi_direction_vs_p50c": round(float(abs(uw @ up) / np.linalg.norm(uw)), 4),
                   "random_baseline": round(1 / np.sqrt(len(up)), 4)}
            if str(li) in diag["neurons"]:
                rec["neuron_overlap"] = f"{len(set(diag['neurons'][str(li)]) & set(cols.tolist()))}/{len(cols)}"
            exam[f"layer_{li}"] = rec
    res["exam_vs_p50c"] = exam
    res["timing"] = timing_summary(work)
    (work / "results_discover.json").write_text(json.dumps(res, indent=2))

    print(f"\n{'arm':10s} {'acc':>7s} {'loops':>7s} {'trunc':>7s}")
    for k, v in res["arms"].items():
        print(f"{k:10s} {v['accuracy']:7.1%} {v['loop_killed']:7.1%} {v['truncated']:7.1%}"
              + (f"   edit on {v['edit_token_share']:.0%} of tokens" if "edit_token_share" in v else ""))
    for k, v in res["paired"].items():
        print(f"  {k}: {v['diff']:+.1%}  95% CI [{v['ci95'][0]:+.1%}, {v['ci95'][1]:+.1%}]")
    print("exam vs p50c:", json.dumps(exam, indent=1))
    print("Saved", work / "results_discover.json")


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    p = sub.add_parser("prepare")
    p.add_argument("--work", required=True)
    p.add_argument("--base-dir")
    p.add_argument("--data-dir")
    p.add_argument("--n-elicit", type=int, default=300)
    p.add_argument("--elicit-max-new", type=int, default=8192)
    p.add_argument("--n-dev", type=int, default=64)
    p.add_argument("--dev-max-new", type=int, default=8192)
    p.add_argument("--n-test", type=int, default=200)
    p.add_argument("--max-new", type=int, default=16384)
    p.add_argument("--seed", type=int, default=0)
    g = sub.add_parser("gen")
    g.add_argument("--work", required=True)
    g.add_argument("--model", required=True)
    g.add_argument("--requests", required=True)
    g.add_argument("--out", required=True)
    g.add_argument("--gpu-mem", type=float, default=0.88)
    g.add_argument("--early-stop", action=argparse.BooleanOptionalAction, default=True,
                   help="stop a response where the loop kill rule fires (same graded result, far fewer tokens)")
    d = sub.add_parser("diagnose")
    d.add_argument("--work", required=True)
    d.add_argument("--top-layers", type=int, default=2)
    d.add_argument("--fa-layers", type=int, default=0, help="also add a candidate with this many full-attention layers")
    d.add_argument("--neurons", type=int, default=64)
    d.add_argument("--after-onset", type=int, default=768, help="tokens kept after the first looping token")
    d.add_argument("--max-seq", type=int, default=4096)
    b = sub.add_parser("build")
    b.add_argument("--work", required=True)
    b.add_argument("--alpha", type=float, required=True)
    b.add_argument("--cand", default="top", help="a key of diagnosis.json candidates: top, late, fa, fa3, ...")
    b.add_argument("--out-dir")
    m = sub.add_parser("plan-moi")
    m.add_argument("--work", required=True)
    k = sub.add_parser("pick-alpha")
    k.add_argument("--work", required=True)
    r = sub.add_parser("report")
    r.add_argument("--work", required=True)
    r.add_argument("--p50c-dir")
    a = ap.parse_args()
    fns = {"prepare": cmd_prepare, "gen": cmd_gen, "diagnose": cmd_diagnose, "build": cmd_build,
           "plan-moi": cmd_plan_moi, "pick-alpha": cmd_pick_alpha, "report": cmd_report}
    key = {"gen": lambda: f"gen:{Path(a.out).stem}:{Path(a.model).name}", "build": lambda: f"build:{a.cand}@{a.alpha:g}",
           "diagnose": lambda: "diagnose", "pick-alpha": lambda: "pick-alpha", "prepare": lambda: "prepare"}.get(a.cmd)
    run_timed(a, fns, key() if key else None)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/discover_followup.py
"""Follow-up of discover_loop_fix.py: where does the WSI-via-MoI gain come from, and can a better fix be picked?

The first run (benchmarks: results_discover.json) gave base 9%, WSI fix always on 14%, WSI fix via MoI 40%, p50c 59.5%.
Three checks, reusing that run's files (restored from Google Drive):
  placebo   the MoI arm's switch itself (cut at the router's firing point, continue with a new seed and every prefix
            token penalised) may break loops without any edit. Same continuation requests, BASE weights.
  retune    the first tuning used 8k-token dev responses while test allows 16k, so strong edits that stopped looping
            but needed longer answers were scored as truncated and the weakest edit won. Re-tune on the same dev
            problems at 16k, over three layer candidates: top, late, and fa (most separated full-attention layers;
            an architecture prior, nothing from p50c).
  report    test arms: base, WSI v1 fixed/MoI, placebo MoI, WSI v2 fixed/MoI, p50c; paired CIs; exam vs p50c.

    python discover_followup.py make-dev16 --work W
    python discover_followup.py pick --work W
    python discover_followup.py report --work W --p50c-dir P
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent))
import discover_loop_fix as dl  # noqa: E402
import moi_loop_router_vllm as lr  # noqa: E402


def cmd_make_dev16(a) -> None:
    work = Path(a.work)
    reqs = lr.read_jsonl(work / "requests_dev.jsonl")
    for r in reqs:
        r["max_tokens"] = a.max_new
    lr.write_jsonl(work / "requests_dev16.jsonl", reqs)
    print(f"requests_dev16.jsonl: {len(reqs)} dev problems at {a.max_new} new tokens")


def cmd_pick(a) -> None:
    work = Path(a.work)
    res = {}
    for f in sorted(work.glob("outputs_dev16_*_a*.jsonl")):
        if f.name == "outputs_dev16_base.jsonl":
            continue
        cand, al = f.stem[len("outputs_dev16_"):].rsplit("_a", 1)
        s = dl.score(work, "requests_dev16.jsonl", f.name)
        res[(cand, float(al))] = {k: v for k, v in s.items() if k != "rows"}
        print(f"  {cand:4s} alpha={float(al):g}: acc={s['accuracy']:.1%} loops={s['loop_killed']:.1%} "
              f"trunc={s['truncated']:.1%}")
    base = dl.score(work, "requests_dev16.jsonl", "outputs_dev16_base.jsonl")
    print(f"  base      : acc={base['accuracy']:.1%} loops={base['loop_killed']:.1%} trunc={base['truncated']:.1%}")
    best = max(res, key=lambda k: (res[k]["accuracy"], -res[k]["loop_killed"]))
    diag = json.loads((work / "diagnosis.json").read_text())
    out = {"cand": best[0], "alpha": best[1], "layers": diag["candidates"][best[0]],
           "candidates": diag["candidates"],
           "dev16": {"base": {k: v for k, v in base.items() if k != "rows"},
                     "grid": {f"{c}@{al:g}": v for (c, al), v in res.items()}},
           "fix_dir": str(work / f"wsi-fix-{best[0]}-a{best[1]:g}")}
    v = a.version
    (work / ("followup.json" if v == 2 else f"followup{v}.json")).write_text(json.dumps(out, indent=2))
    (work / f"chosen_fix{v}.txt").write_text(out["fix_dir"])
    print(f"chosen v{v}: {best[0]} layers {out['layers']}, alpha {best[1]:g}")


def moi_arm(work: Path, cont_file: str, reqs: dict, base_rows: dict, tok, eos: set) -> dict:
    """MoI composite: the base response until the router fires, then the continuation in `cont_file`."""
    plan = {r["id"]: r for r in lr.read_jsonl(work / "requests_test_moi.jsonl")}
    cont = {o["id"]: o for o in lr.read_jsonl(work / cont_file)}
    rows = {}
    for i in reqs:
        if i in plan:
            if i not in cont:
                raise SystemExit(f"{cont_file} lacks id {i}")
            pre = plan[i]["prompt_ids"][len(reqs[i]["prompt_ids"]) :]
            rows[i] = lr.grade(pre + cont[i]["token_ids"], cont[i]["finish_reason"], reqs[i]["gold"], tok, eos,
                               reqs[i]["max_tokens"])
        else:
            rows[i] = base_rows[i]
    return rows


def summary(rows: dict) -> dict:
    n = len(rows)
    return {"n": n, "accuracy": sum(g["correct"] for g in rows.values()) / n,
            "loop_killed": sum(g["looped_killed"] for g in rows.values()) / n,
            "truncated": sum(g["truncated"] for g in rows.values()) / n}


def cmd_report(a) -> None:
    from transformers import AutoTokenizer

    work = Path(a.work)
    cfg = dl.cfg_load(work)
    tok = AutoTokenizer.from_pretrained(cfg["base_dir"])
    eos, reqs = set(cfg["eos"]), {r["id"]: r for r in lr.read_jsonl(work / "requests_test.jsonl")}
    rows: dict = {}
    for name, f in {"base": "outputs_test_base.jsonl", "wsi_fixed": "outputs_test_wsi.jsonl",
                    "wsi2_fixed": "outputs_test_wsi2.jsonl", "wsi3_fixed": "outputs_test_wsi3.jsonl",
                    "wsi4_fixed": "outputs_test_wsi4.jsonl",
                    "p50c": "outputs_test_p50c.jsonl"}.items():
        if (work / f).exists():
            rows[name] = {r["id"]: r for r in dl.score(work, "requests_test.jsonl", f, tok)["rows"]}
    for name, f in {"wsi_moi": "outputs_test_moi.jsonl", "placebo_moi": "outputs_test_placebo.jsonl",
                    "wsi2_moi": "outputs_test_moi2.jsonl", "wsi3_moi": "outputs_test_moi3.jsonl",
                    "wsi4_moi": "outputs_test_moi4.jsonl"}.items():
        if (work / f).exists():
            rows[name] = moi_arm(work, f, reqs, rows["base"], tok, eos)
    ids = sorted(reqs)
    acc = {k: [bool(v[i]["correct"]) for i in ids] for k, v in rows.items()}
    pairs = [("placebo_moi", "base"), ("wsi_moi", "placebo_moi"), ("wsi2_moi", "placebo_moi"),
             ("wsi2_fixed", "base"), ("wsi2_fixed", "wsi_fixed"), ("wsi2_moi", "wsi_moi"), ("wsi2_fixed", "p50c"),
             ("wsi3_fixed", "base"), ("wsi3_fixed", "wsi2_fixed"), ("wsi3_moi", "placebo_moi"), ("wsi3_moi", "wsi2_moi"),
             ("wsi3_fixed", "p50c"), ("wsi3_moi", "p50c"),
             ("wsi4_fixed", "base"), ("wsi4_fixed", "wsi3_fixed"), ("wsi4_moi", "placebo_moi"), ("wsi4_moi", "wsi3_moi"),
             ("wsi4_fixed", "p50c"), ("wsi4_moi", "p50c")]
    fu = json.loads((work / "followup.json").read_text()) if (work / "followup.json").exists() else {}
    fu3 = json.loads((work / "followup3.json").read_text()) if (work / "followup3.json").exists() else None
    res = {"arms": {k: summary(v) for k, v in rows.items()},
           "paired": {f"{x}_vs_{y}": lr.paired_ci(acc[x], acc[y]) for x, y in pairs if x in acc and y in acc},
           "v2_choice": fu}
    if fu3:
        res["v3_choice"] = fu3
    fu4 = json.loads((work / "followup4.json").read_text()) if (work / "followup4.json").exists() else None
    if fu4:
        res["v4_choice"] = fu4  # v3 layers, alpha grid extended to 8 and 10

    exam: dict = {"p50c_layers": [15, 19], "v2_layers": fu.get("layers"),
                  "v3_layers": fu3.get("layers") if fu3 else None,
                  "candidates": (fu3 or fu).get("candidates")}
    if a.p50c_dir:
        from safetensors import safe_open

        diag = json.loads((work / "diagnosis.json").read_text())
        z = np.load(work / "wsi_directions.npz")
        base = Path(cfg["base_dir"])
        for li in (15, 19):
            dW = None
            for f in sorted(Path(a.p50c_dir).glob("*.safetensors")):
                with safe_open(str(f), "pt") as fe, safe_open(str(base / f.name), "pt") as fb:
                    try:
                        k = dl.down_key(list(fe.keys()), li)
                    except SystemExit:
                        continue
                    dW = (fe.get_tensor(k).double() - fb.get_tensor(k).double()).numpy()
            if dW is None:
                continue
            cols = np.where(np.abs(dW).sum(0) > 0)[0]
            up = np.linalg.svd(dW[:, cols], full_matrices=False)[0][:, 0]
            uw = z[f"all_u{li}"].astype(np.float64)
            rec = {"abs_cos_wsi_direction_vs_p50c": round(float(abs(uw @ up) / np.linalg.norm(uw)), 4),
                   "random_baseline": round(1 / np.sqrt(len(up)), 4)}
            if str(li) in diag["neurons"]:
                ov = len(set(diag["neurons"][str(li)]) & set(cols.tolist()))
                rec["neuron_overlap"] = f"{ov}/{len(cols)}"
                rec["neuron_overlap_random_expectation"] = round(len(cols) * a.neurons / dW.shape[1], 2)
            exam[f"layer_{li}"] = rec
    res["exam_vs_p50c"] = exam
    res["timing"] = dl.timing_summary(work)
    (work / a.out).write_text(json.dumps(res, indent=2))

    print(f"\n{'arm':12s} {'acc':>7s} {'loops':>7s} {'trunc':>7s}")
    for k, v in res["arms"].items():
        print(f"{k:12s} {v['accuracy']:7.1%} {v['loop_killed']:7.1%} {v['truncated']:7.1%}")
    for k, v in res["paired"].items():
        print(f"  {k}: {v['diff']:+.1%}  95% CI [{v['ci95'][0]:+.1%}, {v['ci95'][1]:+.1%}]  "
              f"({v['a_only']} vs {v['b_only']})")
    print("exam vs p50c:", json.dumps(exam, indent=1))
    print("Saved", work / a.out)


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    sub = ap.add_subparsers(dest="cmd", required=True)
    m = sub.add_parser("make-dev16")
    m.add_argument("--work", required=True)
    m.add_argument("--max-new", type=int, default=16384)
    p = sub.add_parser("pick")
    p.add_argument("--work", required=True)
    p.add_argument("--version", type=int, default=2, help="2 -> followup.json, 3 -> followup3.json")
    r = sub.add_parser("report")
    r.add_argument("--work", required=True)
    r.add_argument("--p50c-dir")
    r.add_argument("--neurons", type=int, default=64)
    r.add_argument("--out", default="results_discover_followup.json")
    a = ap.parse_args()
    fns = {"make-dev16": cmd_make_dev16, "pick": cmd_pick, "report": cmd_report}
    dl.run_timed(a, fns, {"pick": f"pick:v{getattr(a, 'version', 2)}", "make-dev16": "make-dev16"}.get(a.cmd))


if __name__ == "__main__":
    main()

In [ ]:
# 0. modelo base + recuperar las corridas anteriores desde Drive
!python /content/discover_loop_fix.py prepare --work {W} --n-elicit {N_ELICIT} --elicit-max-new {ELICIT_MAX} --n-dev {N_DEV} --dev-max-new {DEV_MAX} --n-test {N_TEST} --max-new {MAX_NEW}
restaurar()
need = ["diagnosis.json", "wsi_directions.npz", "followup.json", "followup3.json", "requests_test_moi.jsonl",
        "outputs_test_base.jsonl", "outputs_test_placebo.jsonl", "outputs_test_wsi3.jsonl", "outputs_test_moi3.jsonl",
        "outputs_test_p50c.jsonl", "outputs_dev16_base.jsonl", "outputs_dev16_fa3_a6.jsonl", "requests_dev16.jsonl"]
falta = [f for f in need if not hecho(f)]
assert not falta, f"no encuentro las corridas anteriores en Drive: faltan {falta}"
assert "fa3" in json.load(open(f"{W}/diagnosis.json"))["candidates"], "falta el grupo fa3 de v3 en diagnosis.json"
print("corridas anteriores recuperadas; capas de v3:", json.load(open(f"{W}/followup3.json"))["layers"])

In [ ]:
# 1. las mismas 3 capas con α 8 y 10, en los 64 problemas de dev a 16k tokens; luego elegir entre todo lo probado
for a in ALPHAS4:
    if not hecho(f"outputs_dev16_fa3_a{a}.jsonl"):
        !python /content/discover_loop_fix.py build --work {W} --cand fa3 --alpha {a}
        !python /content/discover_loop_fix.py gen --work {W} --model {W}/wsi-fix-fa3-a{a} --requests requests_dev16.jsonl --out outputs_dev16_fa3_a{a}.jsonl
        guardar()
        !rm -rf {W}/wsi-fix-fa3-a{a}
!python /content/discover_followup.py pick --work {W} --version 4
guardar()

In [ ]:
# 2. test solo si cambió el ganador
fu3, fu4 = json.load(open(f"{W}/followup3.json")), json.load(open(f"{W}/followup4.json"))
C4, A4, FIX4 = fu4["cand"], fu4["alpha"], fu4["fix_dir"]
if (C4, float(A4)) == (fu3["cand"], float(fu3["alpha"])):
    print(f"α {fu3['alpha']:g} sigue ganando: no hace falta otro test")
else:
    print("nuevo ganador:", C4, fu4["layers"], "alpha", A4)
    if not hecho("outputs_test_wsi4.jsonl", "outputs_test_moi4.jsonl") and not os.path.exists(FIX4):
        !python /content/discover_loop_fix.py build --work {W} --cand {C4} --alpha {A4}
    if not hecho("outputs_test_wsi4.jsonl"):
        !python /content/discover_loop_fix.py gen --work {W} --model {FIX4} --requests requests_test.jsonl --out outputs_test_wsi4.jsonl
        guardar()
    if not hecho("outputs_test_moi4.jsonl"):
        !python /content/discover_loop_fix.py gen --work {W} --model {FIX4} --requests requests_test_moi.jsonl --out outputs_test_moi4.jsonl
        guardar()

In [ ]:
# 3. reporte (el examen contra p50c no cambia: ya está en results_discover_v3.json)
!python /content/discover_followup.py report --work {W} --out results_discover_v4.json
guardar()
r = json.load(open(f"{W}/results_discover_v4.json"))
print("ajuste (dev, 64 problemas):")
for k, v in r["v4_choice"]["dev16"]["grid"].items():
    print(f"  {k:8s} acc {v['accuracy']:.1%}  loops {v['loop_killed']:.1%}  trunc {v['truncated']:.1%}")
print("tiempos:", r.get("timing", {}).get("discovery_min"), "min de descubrimiento (solo los pasos de esta corrida)")

In [ ]:
from google.colab import files
files.download(f"{W}/results_discover_v4.json")

In [ ]:
# Apagar el runtime al terminar (ahorra unidades de cómputo). No depende de las celdas de arriba:
# funciona aunque el runtime se haya reiniciado. Solo apaga si el resultado ya está en Google Drive.
AUTO_APAGAR = True
import os, time
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"):
    drive.mount("/content/drive")
if AUTO_APAGAR and os.path.exists("/content/drive/MyDrive/wsi_runs/discover_loop_fix/results_discover_v4.json"):
    time.sleep(60)
    from google.colab import runtime
    runtime.unassign()
else:
    print("No apago: el resultado no está en Drive todavía.")